# Hunyuan3D-2 — Live Gradio App (Image → 3D)

Runs the official Hunyuan3D-2 Gradio demo on Kaggle GPU and exposes it via a **public gradio.live link** (valid while this kernel session stays alive — up to Kaggle's session limit, ~9-12h on the free tier). Upload any image in the UI and get a downloadable `.glb` back.

Enable **Accelerator: GPU T4 x2**, **Internet: on** in Notebook Settings before running.

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    raise SystemExit('No GPU detected — enable GPU T4 x2 in Notebook Settings > Accelerator, then re-run.')

In [ ]:
%cd /kaggle/working
!git clone --depth 1 https://github.com/Tencent-Hunyuan/Hunyuan3D-2.git
%cd Hunyuan3D-2
!pip install -q -r requirements.txt
!pip install -q gradio fastapi uvicorn

In [ ]:
import subprocess

def try_build(path, cmd):
    try:
        subprocess.run(cmd, cwd=path, shell=True, check=True)
        print(f'built OK: {path}')
    except subprocess.CalledProcessError as e:
        print(f'skipping optional build ({path}): {e}')

try_build('/kaggle/working/Hunyuan3D-2/hy3dgen/texgen/custom_rasterizer', 'python3 setup.py install')

## Patch the app to open a public share link

The stock `gradio_app.py` mounts Gradio into a local-only FastAPI/uvicorn server (fine for a machine with a browser, useless inside a Kaggle sandbox). This swaps that final line for `demo.launch(share=True, ...)`, which opens a public `*.gradio.live` tunnel instead.

In [ ]:
path = '/kaggle/working/Hunyuan3D-2/gradio_app.py'
with open(path) as f:
    src = f.read()

old = 'app = gr.mount_gradio_app(app, demo, path="/")\n    uvicorn.run(app, host=args.host, port=args.port, workers=1)'
new = 'demo.queue().launch(share=True, server_name=args.host, server_port=args.port, show_error=True)'
assert old in src, 'expected launch block not found — upstream file may have changed'
src = src.replace(old, new)

with open(path, 'w') as f:
    f.write(src)
print('patched gradio_app.py for public sharing')

In [ ]:
%cd /kaggle/working/Hunyuan3D-2
# Fast/light config: the mini-turbo shape model, texture generation disabled
# (keeps startup fast and avoids the texgen pipeline's heavier VRAM/build needs).
!python gradio_app.py --model_path tencent/Hunyuan3D-2mini --subfolder hunyuan3d-dit-v2-mini-turbo --disable_tex --low_vram_mode --port 8080 --host 0.0.0.0